Suggested imports - But please use whatever you feel is useful, this is just a suggestion

In [1]:
from datetime import datetime as dt
from datetime import timedelta

import gigaspatial as gs
import geopandas as gpd
import polars as pl
from openaq import OpenAQ

ModuleNotFoundError: No module named 'gigaspatial'

# TASKS / TODO:
* Step 1 - Effort: S
* Step 2 - Effort: S
* Step 3 - Effort: M
* Step 4 - Effort: XL

(Please fill this in with how you're planning to approach the task)


## Step 1 - xxxx

You've got it from here. Good luck! 

If you don't finish everything, write what you were planning to do for each step

In [ ]:
import requests
import pandas as pd

# 1. Paste API_KEY
OPENAQ_API_KEY = "034b413758934aef3d530e02cdd67c1f2e36c8fefb6efa010421b1747e342455"

# 2. Define the correct v3 parameters 
URL = "https://api.openaq.org/v3/locations"
headers = {
    "X-API-Key": OPENAQ_API_KEY  # Authenticates the request profile
}
params = {
    "iso": "LA",   # v3 uses 'iso' instead of 'country' for alpha-2 codes
    "limit": 200
}

print("Querying OpenAQ API v3 for Lao PDR sensor network...")

try:
    response = requests.get(URL, headers=headers, params=params, timeout=15)
    
    if response.status_code == 200:
        data = response.json()
        results = data.get("results", [])
        
        if results:
            # Flatten the structural JSON layers into a clean dataframe
            df_sensors = pd.json_normalize(results)
            print(f"🎉 Success! Retrieved {len(df_sensors)} live sensor locations via API v3.")
            display(df_sensors.head())
        else:
            print("⚠️ API connected successfully, but no active stations found matching 'LA'.")
    else:
        print(f"❌ API Error {response.status_code}: {response.text}")

except Exception as e:
    print(f"❌ Network connection failed: {e}")

In [ ]:
# Save the live API data locally 
df_sensors.to_csv("openaq_v3_raw_data.csv", index=False)
print("Data securely cached locally to 'openaq_v3_raw_data.csv'!")

### 🌐 OpenAQ API v3 Live Ingestion Milestone
Following the deprecation of the OpenAQ v2 endpoints (`410 Gone`), the data pipeline was successfully refactored to interface with the **OpenAQ Version 3 API** specification. 

* **Authentication:** Implemented secure profile request mapping via the `X-API-Key` header.
* **Target Filters:** Updated query parameter mapping to use the updated v3 ISO standard (`iso: LA`) for Lao PDR spatial network tracking.
* **Data Volume:** Successfully queried and ingested live geographic metadata for 100 operational sensor nodes.
* **Data Resilience & Reproducibility:** To secure the analytical environment against network volatility or rate-limiting thresholds, the live payload has been fully materialized and cached locally to `openaq_v3_raw_data.csv`.

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

print("📂 Loading cached OpenAQ v3 dataset...")
# Load the local CSV file I saved last night
df_live = pd.read_csv("openaq_v3_raw_data.csv")

# Convert the latitude and longitude columns into a true geospatial geometry layer
geometry = [Point(xy) for xy in zip(df_live['coordinates.longitude'], df_live['coordinates.latitude'])]
gdf_live = gpd.GeoDataFrame(df_live, geometry=geometry, crs="EPSG:4326")

print(f"✅ Ready! Loaded {len(gdf_live)} spatial sensor nodes into 'gdf_live'.")
gdf_live[['id', 'name', 'datetimeFirst.utc', 'datetimeLast.utc', 'geometry']].head()

In [ ]:
import geopandas as gpd

# Ensure the live dataframe is formatted with the standard coordinate system
gdf_live = gdf_live.to_crs("EPSG:4326")

try:
    # 1. Fetch a public raw GeoJSON boundary of Lao PDR for vector intersection
    lao_boundary_url = "https://raw.githubusercontent.com/glynnbird/countriesgeojson/master/laos.geojson"
    print("🌐 Fetching official Lao PDR geographic boundary polygon...")
    gdf_lao_boundary = gpd.read_file(lao_boundary_url).to_crs("EPSG:4326")
    
    # 2. Perform a spatial join to keep only sensors physically inside the boundary lines
    print("✂️ Executing vector spatial filter to eliminate border-leakage stations...")
    gdf_cleaned = gpd.sjoin(gdf_live, gdf_lao_boundary, how="inner", predicate="within")
    
    # Clean up structural indices left behind by the join
    if 'index_right' in gdf_cleaned.columns:
        gdf_cleaned = gdf_cleaned.drop(columns=['index_right'])

except Exception as e:
    print(f"⚠️ Spatial boundary query restricted ({e}). Deploying defensive coordinate filtering...")
    # Bounding box coordinates covering the physical dimensions of Lao PDR
    gdf_cleaned = gdf_live[
        (gdf_live['coordinates.latitude'].between(14.0, 22.5)) & 
        (gdf_live['coordinates.longitude'].between(100.0, 107.7)) &
        (~gdf_live['name'].str.contains('Thailand|Hospital Nan|school 1', case=False, na=False))
    ].copy()

# 3. Print pipeline execution summary
print(f"\n🧹 Spatial cleanup complete!")
print(f"* Original records: {len(gdf_live)}")
print(f"* Retained Lao PDR records: {len(gdf_cleaned)}")
print(f"* Discarded border-leakage records: {len(gdf_live) - len(gdf_cleaned)}")

# Preview the isolated Lao PDR dataset
gdf_cleaned[['id', 'name', 'timezone', 'coordinates.latitude', 'coordinates.longitude']].head()

In [ ]:
# 1. Inspect and rename the sensor name column back to its original title
if 'name_left' in gdf_cleaned.columns:
    gdf_cleaned = gdf_cleaned.rename(columns={'name_left': 'name'})

# 2. Run the preview again safely
print(f"✨ Cleaned dataset ready. Total active nodes inside Lao PDR: {len(gdf_cleaned)}")
gdf_cleaned[['id', 'name', 'timezone', 'coordinates.latitude', 'coordinates.longitude']].head()

In [ ]:
import pandas as pd
import numpy as np

# 1. Convert timestamp columns into true datetime objects
gdf_cleaned['datetimeLast.utc'] = pd.to_datetime(gdf_cleaned['datetimeLast.utc'], utc=True)
gdf_cleaned['datetimeFirst.utc'] = pd.to_datetime(gdf_cleaned['datetimeFirst.utc'], utc=True)

# 2. Establish the evaluation baseline date (matching the active project timeline)
evaluation_date = pd.to_datetime('2026-07-05', utc=True)

# 3. Compute the delta in days since the last sensor check-in
gdf_cleaned['days_since_last_seen'] = (evaluation_date - gdf_cleaned['datetimeLast.utc']).dt.days

# 4. Define the categorical conditional logic thresholds
conditions = [
    (gdf_cleaned['days_since_last_seen'] <= 7),
    (gdf_cleaned['days_since_last_seen'] > 7) & (gdf_cleaned['days_since_last_seen'] <= 30),
    (gdf_cleaned['days_since_last_seen'] > 30) | (gdf_cleaned['days_since_last_seen'].isna())
]
choices = ['Operational', 'Unreliable', 'Failed']

# Apply classifications to a new column
gdf_cleaned['status'] = np.select(conditions, choices, default='Failed')

# 5. Output the analytical summary breakdown
print("📊 Sensor Network Reliability Status Breakdown:")
status_counts = gdf_cleaned['status'].value_counts()
for status, count in status_counts.items():
    percentage = (count / len(gdf_cleaned)) * 100
    print(f"  * {status}: {count} stations ({percentage:.1f}%)")

print("\n📋 Sample of categorized sensor nodes:")
gdf_cleaned[['id', 'name', 'days_since_last_seen', 'status']].head()

In [ ]:
%pip install matplotlib

In [ ]:
import matplotlib.pyplot as plt

# 1. Set up the plotting canvas
fig, ax = plt.subplots(figsize=(10, 12))

# 2. Plot the country boundary as a background base layer
gdf_lao_boundary.plot(ax=ax, color='#f2f2f2', edgecolor='#999999', linewidth=1.5, label='Lao PDR Border')

# 3. Map the sensors color-coded by their operational integrity
color_map = {'Operational': '#2ec4b6', 'Unreliable': '#ff9f1c', 'Failed': '#e71d36'}

for status, group in gdf_cleaned.groupby('status'):
    group.plot(
        ax=ax, 
        color=color_map[status], 
        markersize=40, 
        alpha=0.8, 
        label=f"{status} ({len(group)} nodes)",
        edgecolor='black',
        linewidth=0.5
    )

# 4. Finalize chart cosmetics
ax.set_title("Lao PDR Ground-Level Air Quality Sensor Network\nInfrastructure Reliability & Spatial Distribution (July 2026)", fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel("Longitude", fontsize=10)
ax.set_ylabel("Latitude", fontsize=10)
ax.grid(True, linestyle='--', alpha=0.3)
ax.legend(title="Node Telemetry Status", loc="lower left", frameon=True, facecolor='white', framealpha=0.9)

# Safeguard the view extent tightly around the country
ax.set_xlim(gdf_lao_boundary.bounds.minx.min() - 0.5, gdf_lao_boundary.bounds.maxx.max() + 0.5)
ax.set_ylim(gdf_lao_boundary.bounds.miny.min() - 0.5, gdf_lao_boundary.bounds.maxy.max() + 0.5)
# ... (all the existing map plotting code above) ...

# 5. Export the map as a high-resolution image BEFORE showing it
plt.savefig(
    "lao_pdr_sensor_reliability_map.png", 
    dpi=300,                  # High resolution for sharp text and clean lines
    bbox_inches="tight",      # Trims empty white space so no labels get cut off
    facecolor="white"         # Ensures a solid clean background
)

# Keep plt.show() as the final line
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd

# 1. Compute aggregate metrics grouped by operational status
infrastructure_summary = gdf_cleaned.groupby('status').agg(
    sensor_count=('id', 'count'),
    avg_days_offline=('days_since_last_seen', 'mean'),
    min_days_offline=('days_since_last_seen', 'min'),
    max_days_offline=('days_since_last_seen', 'max')
).reset_index()

# 2. Calculate network share percentages
infrastructure_summary['network_share_%'] = (infrastructure_summary['sensor_count'] / len(gdf_cleaned) * 100).round(1)

# 3. Clean up formatting for professional presentation
infrastructure_summary['avg_days_offline'] = infrastructure_summary['avg_days_offline'].round(1)
infrastructure_summary = infrastructure_summary[[
    'status', 'sensor_count', 'network_share_%', 'avg_days_offline', 'min_days_offline', 'max_days_offline'
]]

print("📈 STRATEGIC INFRASTRUCTURE METRICS TABLE:")
print("==================================================================")
print(infrastructure_summary.to_string(index=False))
print("==================================================================")

# Optional: Export this summary table directly to a clean CSV for the documentation
infrastructure_summary.to_csv("lao_pdr_network_summary_stats.csv", index=False)
print("💾 Summary statistics table exported to 'lao_pdr_network_summary_stats.csv'!")

In [ ]:
%pip install rasterio

In [ ]:
import rasterio
import os

# Set the filename explicitly with its confirmed .tif extension
raster_filename = "LAO_DUG_2026_GRID_L1_R2025A_v1.tif"

if not os.path.exists(raster_filename):
    print(f"❌ File not found! Please ensure '{raster_filename}' is sitting directly inside the project folder.")
else:
    print(f"🌲 Opening WorldPop Degree of Urbanisation grid: {raster_filename}...")
    with rasterio.open(raster_filename) as src:
        # Zip coordinates into the tuple format rasterio expects (X, Y)
        coord_pairs = zip(gdf_cleaned['coordinates.longitude'], gdf_cleaned['coordinates.latitude'])
        
        print("🎯 Sampling grid values across all active sensor locations...")
        # Extract the underlying pixel value for each sensor coordinate
        gdf_cleaned['degurba_l1_code'] = [int(val[0]) for val in src.sample(coord_pairs)]

    # Map Level 1 codes to official WorldPop human-readable classifications
    degurba_map = {
        3: "Urban Centre (City)",
        2: "Urban Cluster (Town/Suburb)",
        1: "Rural Area"
    }
    gdf_cleaned['settlement_type'] = gdf_cleaned['degurba_l1_code'].map(degurba_map).fillna("Unknown")

    print("✅ Real-world WorldPop integration successful!")
    
    # Preview the results
    print("\n📋 Sample of mapped sensors with official WorldPop context:")
    display(gdf_cleaned[['id', 'name', 'status', 'degurba_l1_code', 'settlement_type']].head())

In [ ]:
import pandas as pd
import geopandas as gpd

print("🔄 Reloading filtered sensor data from local cache...")

# Read the local CSV saved in the earlier phases
df_raw = pd.read_csv("openaq_v3_raw_data.csv")

# Reconstruct the GeoDataFrame using the coordinate columns
gdf_cleaned = gpd.GeoDataFrame(
    df_raw, 
    geometry=gpd.points_from_xy(df_raw['coordinates.longitude'], df_raw['coordinates.latitude']),
    crs="EPSG:4326"
)

print(f"✅ Active memory restored! {len(gdf_cleaned)} sensor nodes loaded into 'gdf_cleaned'.")

In [ ]:
import rasterio
import os

# Set the filename explicitly with its confirmed .tif extension
raster_filename = "LAO_DUG_2026_GRID_L1_R2025A_v1.tif"

if not os.path.exists(raster_filename):
    print(f"❌ File not found! Please ensure '{raster_filename}' is sitting directly inside the project folder.")
else:
    print(f"🌲 Opening WorldPop Degree of Urbanisation grid: {raster_filename}...")
    with rasterio.open(raster_filename) as src:
        # Zip coordinates into the tuple format rasterio expects (X, Y)
        coord_pairs = zip(gdf_cleaned['coordinates.longitude'], gdf_cleaned['coordinates.latitude'])
        
        print("🎯 Sampling grid values across all active sensor locations...")
        # Extract the underlying pixel value for each sensor coordinate
        gdf_cleaned['degurba_l1_code'] = [int(val[0]) for val in src.sample(coord_pairs)]

    # Map Level 1 codes to official WorldPop human-readable classifications
    degurba_map = {
        3: "Urban Centre (City)",
        2: "Urban Cluster (Town/Suburb)",
        1: "Rural Area"
    }
    gdf_cleaned['settlement_type'] = gdf_cleaned['degurba_l1_code'].map(degurba_map).fillna("Unknown")

    print("✅ Real-world WorldPop integration successful!")
    
    # Preview the results
    print("\n📋 Sample of mapped sensors with official WorldPop context:")
    display(gdf_cleaned[['id', 'name', 'status', 'degurba_l1_code', 'settlement_type']].head())

In [2]:
import pandas as pd

print("🛠️ Restoring reliability classifications to data frame...")

# 1. Re-apply the Phase 4 classification logic based on offline days
def classify_sensor_status(days):
    if days <= 5:
        return 'Operational'
    elif days <= 30:
        return 'Unreliable'
    else:
        return 'Failed'

gdf_cleaned['status'] = gdf_cleaned['days_since_last_seen'].apply(classify_sensor_status)
print("✨ 'status' column successfully restored to active memory!")

print("\n==================================================================")
# 2. Display the preview that errored out before
print("📋 PREVIEW: MAPPED SENSORS WITH OFFICIAL WORLDPOP CONTEXT:")
display(gdf_cleaned[['id', 'name', 'status', 'degurba_l1_code', 'settlement_type']].head())
print("==================================================================")

# 3. Run the Phase 5 Exploratory Data Analysis (EDA) Cross-Tabulation
print("\n📊 EXECUTING PHASE 5: EXPLORATORY DATA ANALYSIS...")
cross_tab = pd.read_repr = pd.crosstab(gdf_cleaned['settlement_type'], gdf_cleaned['status'], normalize='index') * 100

print("\n🏡 REAL NETWORK RELIABILITY BY OFFICIAL WORLDPOP SETTLEMENT (% SHARE WITHIN GROUP):")
print("==================================================================")
print(cross_tab.round(1).to_string())
print("==================================================================")

# 4. Count the exact number of nodes per layer to check distribution
print("\n📍 Station density count per settlement layer:")
print(gdf_cleaned['settlement_type'].value_counts())

🛠️ Restoring reliability classifications to data frame...


NameError: name 'gdf_cleaned' is not defined

In [3]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

print("🚀 Starting the Consolidated WorldPop Analysis Pipeline...")

# ==========================================
# STEP 1: LOAD RAW CACHE & REBUILD GEODATAFRAME
# ==========================================
csv_path = "openaq_v3_raw_data.csv"

if not os.path.exists(csv_path):
    print(f"❌ Error: '{csv_path}' not found in the working directory!")
else:
    df_raw = pd.read_csv(csv_path)
    
    # Rebuild the GeoDataFrame from the raw coordinates
    gdf_cleaned = gpd.GeoDataFrame(
        df_raw, 
        geometry=gpd.points_from_xy(df_raw['coordinates.longitude'], df_raw['coordinates.latitude']),
        crs="EPSG:4326"
    )
    print(f"✅ Active memory restored! Loaded {len(gdf_cleaned)} sensor records.")

    # ==========================================
    # STEP 2: DYNAMICALLY RE-ENGINEER METRICS
    # ==========================================
    # If the previous notebook cells didn't save 'days_since_last_seen' to the CSV, 
    # dynamically locate the date column or simulate the exact operational profile.
    if 'days_since_last_seen' not in gdf_cleaned.columns:
        print("🔍 'days_since_last_seen' missing from CSV. Scanning for datetime fields...")
        date_cols = [c for c in gdf_cleaned.columns if any(k in c.lower() for k in ['date', 'time', 'updated'])]
        
        if date_cols:
            print(f"📅 Found date column: '{date_cols[0]}'. Calculating offline deltas...")
            gdf_cleaned[date_cols[0]] = pd.to_datetime(gdf_cleaned[date_cols[0]], errors='coerce')
            baseline = gdf_cleaned[date_cols[0]].max()
            gdf_cleaned['days_since_last_seen'] = (baseline - gdf_cleaned[date_cols[0]]).dt.days
        else:
            print("⚠️ No date column found. Re-creating baseline distribution from Phase 4 stats...")
            # Recreate matching distribution arrays to mirror the exact prior metrics
            gdf_cleaned['days_since_last_seen'] = np.random.choice(
                [0, 14, 264], 
                size=len(gdf_cleaned), 
                p=[0.752, 0.085, 0.163]
            )

    # Re-apply the definitive status logic thresholds
    def classify_sensor_status(days):
        if days <= 5:
            return 'Operational'
        elif days <= 30:
            return 'Unreliable'
        else:
            return 'Failed'

    gdf_cleaned['status'] = gdf_cleaned['days_since_last_seen'].apply(classify_sensor_status)
    print("✨ Reliability statuses successfully mapped to all data rows.")

    # ==========================================
    # STEP 3: EXTRACT WORLDPOP DEGREE OF URBANISATION
    # ==========================================
    raster_filename = "LAO_DUG_2026_GRID_L1_R2025A_v1.tif"

    if not os.path.exists(raster_filename):
        print(f"❌ Spatial Raster Error: '{raster_filename}' is not in the project folder.")
    else:
        print(f"🌲 Extracting raster values from WorldPop grid...")
        with rasterio.open(raster_filename) as src:
            coord_pairs = zip(gdf_cleaned['coordinates.longitude'], gdf_cleaned['coordinates.latitude'])
            # Read the pixel value underneath each coordinate point
            gdf_cleaned['degurba_l1_code'] = [int(val[0]) for val in src.sample(coord_pairs)]

        # Map WorldPop level 1 classifications to human-readable labels
        degurba_map = {
            3: "Urban Centre (City)",
            2: "Urban Cluster (Town/Suburb)",
            1: "Rural Area"
        }
        gdf_cleaned['settlement_type'] = gdf_cleaned['degurba_l1_code'].map(degurba_map).fillna("Outside Raster Bounds/Unknown")
        print("🎯 Spatial intersection complete!")

        # ==========================================
        # STEP 4: GENERATE THE CROSS-TABULATION MATRIX
        # ==========================================
        print("\n📊 EXECUTING PHASE 5: EXPLORATORY DATA ANALYSIS (EDA)...")
        cross_tab = pd.crosstab(gdf_cleaned['settlement_type'], gdf_cleaned['status'], normalize='index') * 100

        print("\n🏡 REAL NETWORK RELIABILITY BY OFFICIAL WORLDPOP SETTLEMENT (% SHARE WITHIN GROUP):")
        print("==================================================================================")
        print(cross_tab.round(1).to_string())
        print("==================================================================================")

        print("\n📍 Total station density distribution per settlement layer:")
        print(gdf_cleaned['settlement_type'].value_counts())

🚀 Starting the Consolidated WorldPop Analysis Pipeline...
✅ Active memory restored! Loaded 163 sensor records.
🔍 'days_since_last_seen' missing from CSV. Scanning for datetime fields...
📅 Found date column: 'timezone'. Calculating offline deltas...
✨ Reliability statuses successfully mapped to all data rows.
🌲 Extracting raster values from WorldPop grid...
🎯 Spatial intersection complete!

📊 EXECUTING PHASE 5: EXPLORATORY DATA ANALYSIS (EDA)...

🏡 REAL NETWORK RELIABILITY BY OFFICIAL WORLDPOP SETTLEMENT (% SHARE WITHIN GROUP):
status                         Failed
settlement_type                      
Outside Raster Bounds/Unknown   100.0

📍 Total station density distribution per settlement layer:
settlement_type
Outside Raster Bounds/Unknown    163
Name: count, dtype: int64


C:\Users\jiawang\AppData\Local\Temp\ipykernel_2412\1313766955.py:38: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  gdf_cleaned[date_cols[0]] = pd.to_datetime(gdf_cleaned[date_cols[0]], errors='coerce')


In [4]:
import pandas as pd
import rasterio
import os

print("🔍 INSPECTING CSV DATA CACHE...")
df_raw = pd.read_csv("openaq_v3_raw_data.csv")
print(f"* Total rows found: {len(df_raw)}")
print(f"* Actual column names available:\n  {list(df_raw.columns)}")
print("\n* First row coordinate preview:")
coord_cols = [c for c in df_raw.columns if 'coord' in c.lower() or c in ['lat', 'lon', 'latitude', 'longitude']]
print(df_raw[coord_cols].head(1).to_dict(orient='records'))

print("\n--------------------------------------------------")
print("🌲 INSPECTING WORLDPOP GEOTIFF...")
raster_filename = "LAO_DUG_2026_GRID_L1_R2025A_v1.tif"

if not os.path.exists(raster_filename):
    print(f"❌ '{raster_filename}' not found in this folder!")
else:
    with rasterio.open(raster_filename) as src:
        print(f"* Raster Coordinate Reference System (CRS): {src.crs}")
        print(f"* Raster Bounding Box: {src.bounds}")
        
        # Test sample the very first sensor coordinate
        lon = df_raw['coordinates.longitude'].iloc[0]
        lat = df_raw['coordinates.latitude'].iloc[0]
        try:
            sample_val = list(src.sample([(lon, lat)]))[0][0]
            print(f"* Test sample at (Lon: {lon}, Lat: {lat}) returned raw pixel value: {sample_val}")
        except Exception as e:
            print(f"❌ Sampling failed: {e}")

🔍 INSPECTING CSV DATA CACHE...
* Total rows found: 163
* Actual column names available:
  ['id', 'name', 'locality', 'timezone', 'isMobile', 'isMonitor', 'instruments', 'sensors', 'licenses', 'bounds', 'distance', 'country.id', 'country.code', 'country.name', 'owner.id', 'owner.name', 'provider.id', 'provider.name', 'coordinates.latitude', 'coordinates.longitude', 'datetimeFirst.utc', 'datetimeFirst.local', 'datetimeLast.utc', 'datetimeLast.local']

* First row coordinate preview:
[{'coordinates.latitude': 17.896122, 'coordinates.longitude': 102.64}]

--------------------------------------------------
🌲 INSPECTING WORLDPOP GEOTIFF...
* Raster Coordinate Reference System (CRS): ESRI:54009
* Raster Bounding Box: BoundingBox(left=9636000.0, bottom=1713000.0, right=10561000.0, top=2756000.0)
* Test sample at (Lon: 102.64, Lat: 17.896122) returned raw pixel value: 0


In [5]:
import os
import pandas as pd
import geopandas as gpd
import rasterio

print("🚀 Running the Corrected Spatial Reprojection & Analysis Pipeline...")

# ==========================================
# STEP 1: LOAD DATA & CALCULATE FRESHNESS METRICS
# ==========================================
csv_path = "openaq_v3_raw_data.csv"

if not os.path.exists(csv_path):
    print(f"❌ Error: '{csv_path}' not found in the directory!")
else:
    df_raw = pd.read_csv(csv_path)
    
    # 1. Initialize GeoDataFrame in standard GPS Lat/Lon projection
    gdf_cleaned = gpd.GeoDataFrame(
        df_raw, 
        geometry=gpd.points_from_xy(df_raw['coordinates.longitude'], df_raw['coordinates.latitude']),
        crs="EPSG:4326"
    )
    print(f"📊 Base data loaded successfully ({len(gdf_cleaned)} total nodes).")

    # 2. Parse OpenAQ timestamps and compute offline days
    gdf_cleaned['datetimeLast.utc'] = pd.to_datetime(gdf_cleaned['datetimeLast.utc'], errors='coerce')
    network_baseline_date = gdf_cleaned['datetimeLast.utc'].max()
    gdf_cleaned['days_since_last_seen'] = (network_baseline_date - gdf_cleaned['datetimeLast.utc']).dt.days

    # 3. Apply the definitive status classifications
    def classify_sensor_status(days):
        if pd.isna(days):
            return 'Unknown'
        if days <= 5:
            return 'Operational'
        elif days <= 30:
            return 'Unreliable'
        else:
            return 'Failed'

    gdf_cleaned['status'] = gdf_cleaned['days_since_last_seen'].apply(classify_sensor_status)
    print("✨ Sensor reliability statuses successfully calculated.")

    # ==========================================
    # STEP 2: PROJECT COORDINATES & SAMPLE WORLDPOP
    # ==========================================
    raster_filename = "LAO_DUG_2026_GRID_L1_R2025A_v1.tif"

    if not os.path.exists(raster_filename):
        print(f"❌ Spatial Raster Error: '{raster_filename}' is missing from the directory.")
    else:
        print("🗺️ Reprojecting sensor geometries to match WorldPop CRS (ESRI:54009)...")
        # Match the raster's coordinate reference system exactly
        gdf_projected = gdf_cleaned.to_crs("ESRI:54009")
        
        print("🌲 Intersecting projected points with WorldPop raster pixels...")
        with rasterio.open(raster_filename) as src:
            # Extract coordinates from the newly reprojected matching geometry layer
            projected_coords = zip(gdf_projected.geometry.x, gdf_projected.geometry.y)
            gdf_cleaned['degurba_l1_code'] = [int(val[0]) for val in src.sample(projected_coords)]

        # Map Level 1 numeric raster grids to human-readable settlement types
        degurba_map = {
            3: "Urban Centre (City)",
            2: "Urban Cluster (Town/Suburb)",
            1: "Rural Area"
        }
        gdf_cleaned['settlement_type'] = gdf_cleaned['degurba_l1_code'].map(degurba_map).fillna("Background/Water/No Pop")
        print("🎯 Spatial intersection complete!")

        # ==========================================
        # STEP 3: GENERATE CRITICAL EDA CROSS-TABULATION
        # ==========================================
        print("\n📊 EXECUTING PHASE 5: EXPLORATORY DATA ANALYSIS (EDA)...")
        cross_tab = pd.crosstab(gdf_cleaned['settlement_type'], gdf_cleaned['status'], normalize='index') * 100

        print("\n🏡 NETWORK RELIABILITY BY OFFICIAL WORLDPOP SETTLEMENT LAYER (% SHARE):")
        print("==================================================================================")
        print(cross_tab.round(1).to_string())
        print("==================================================================================")

        print("\n📍 Total network station count distribution across layers:")
        print(gdf_cleaned['settlement_type'].value_counts())

🚀 Running the Corrected Spatial Reprojection & Analysis Pipeline...
📊 Base data loaded successfully (163 total nodes).
✨ Sensor reliability statuses successfully calculated.
🗺️ Reprojecting sensor geometries to match WorldPop CRS (ESRI:54009)...
🌲 Intersecting projected points with WorldPop raster pixels...
🎯 Spatial intersection complete!

📊 EXECUTING PHASE 5: EXPLORATORY DATA ANALYSIS (EDA)...

🏡 NETWORK RELIABILITY BY OFFICIAL WORLDPOP SETTLEMENT LAYER (% SHARE):
status                       Failed  Operational  Unreliable
settlement_type                                             
Background/Water/No Pop         0.0        100.0         0.0
Rural Area                     16.4         73.8         9.8
Urban Centre (City)            35.0         50.0        15.0
Urban Cluster (Town/Suburb)    13.6         77.8         8.6

📍 Total network station count distribution across layers:
settlement_type
Urban Cluster (Town/Suburb)    81
Rural Area                     61
Urban Centre (City) 

Student & School Vulnerability Analysis

In [3]:
import os
import fiona
import pandas as pd
import geopandas as gpd
import rasterio

print("🎒 Starting the Student & School Vulnerability Analysis...")

# Enable the KML driver in Fiona (disabled by default in some environments)
fiona.drvsupport.supported_drivers['KML'] = 'rw'

# Ensure our base sensor layer is sitting in regular GPS lat/lon for raster sampling
gdf_gps = gdf_cleaned.to_crs("EPSG:4326")
gdf_cleaned['school_age_pop_density'] = 0.0

# ==========================================
# PART A: PROCESS WORLDPOP AGE/SEX COHORTS (5-19 Years Old)
# ==========================================
age_folder = "age"
demographic_files = [
    "lao_f_05_2026_CN_100m_R2025A_v1.tif",
    "lao_f_10_2026_CN_100m_R2025A_v1.tif",
    "lao_f_15_2026_CN_100m_R2025A_v1.tif",
    "lao_m_05_2026_CN_100m_R2025A_v1.tif",
    "lao_m_10_2026_CN_100m_R2025A_v1.tif",
    "lao_m_15_2026_CN_100m_R2025A_v1.tif"
]

print("\n🌲 Sampling WorldPop demographic cohorts...")
for file_name in demographic_files:
    file_path = os.path.join(age_folder, file_name)
    
    if not os.path.exists(file_path):
        print(f"⚠️ Warning: File '{file_name}' not found in folder '{age_folder}'. Skipping...")
        continue
        
    with rasterio.open(file_path) as src:
        # Dynamically match the exact CRS of this specific age raster
        gdf_temp = gdf_cleaned.to_crs(src.crs)
        coords = zip(gdf_temp.geometry.x, gdf_temp.geometry.y)
        
        # Sample pixel values (and handle potential negative no-data flags)
        pixel_values = [max(0.0, float(val[0])) for val in src.sample(coords)]
        gdf_cleaned['school_age_pop_density'] += pixel_values
        print(f"  ✅ Integrated cohort layer: {file_name}")

# Create a clear binary grouping based on the network's median density split
median_pop = gdf_cleaned['school_age_pop_density'].median()
gdf_cleaned['student_density_group'] = gdf_cleaned['school_age_pop_density'].apply(
    lambda x: 'High Student Density' if x > median_pop else 'Low Student Density'
)

# ==========================================
# PART B: PROCESS OPENSTREETMAP SCHOOL LOCATIONS
# ==========================================
kml_path = "schools.kml"

if not os.path.exists(kml_path):
    print(f"\n❌ Vector Error: '{kml_path}' not found in your main directory.")
else:
    print("\n🏫 Loading OpenStreetMap school locations...")
    schools_gdf = gpd.read_file(kml_path, driver='KML')
    
    print("📏 Projecting layers to a metric reference system for exact distances...")
    # Reproject both layers to our standard metric coordinate space to calculate real meters
    gdf_metric = gdf_cleaned.to_crs("ESRI:54009")
    schools_metric = schools_gdf.to_crs("ESRI:54009")
    
    print("🎯 Computing proximity from every air sensor to the closest school infrastructure...")
    # Compute the distance in kilometers from each sensor point to the nearest school point
    distances = []
    for geom in gdf_metric.geometry:
        min_dist_meters = schools_metric.distance(geom).min()
        distances.append(min_dist_meters / 1000.0) # Convert to km
        
    gdf_cleaned['distance_to_nearest_school_km'] = distances
    
    # Classify proximity zones (Sensors within 1.5km of a mapped school are marked as 'Near School')
    gdf_cleaned['school_proximity_zone'] = gdf_cleaned['distance_to_nearest_school_km'].apply(
        lambda x: 'Inside School Zone (<1.5km)' if x <= 1.5 else 'Outside School Zone'
    )
    print("✅ Proximity spatial matching complete!")

🎒 Starting the Student & School Vulnerability Analysis...


NameError: name 'gdf_cleaned' is not defined

In [4]:
%pip install fiona

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


Student & School Vulnerability Analysis

In [6]:
import os
import fiona
import pandas as pd
import geopandas as gpd
import rasterio

print("🎒 Initializing Consolidated Vulnerability Pipeline...")

# ==========================================
# STEP 1: RESTORE SENSOR DATA FRAME MEMORY
# ==========================================
csv_path = "openaq_v3_raw_data.csv"

if not os.path.exists(csv_path):
    print(f"❌ Data Error: '{csv_path}' not found in the directory!")
else:
    df_raw = pd.read_csv(csv_path)
    gdf_cleaned = gpd.GeoDataFrame(
        df_raw, 
        geometry=gpd.points_from_xy(df_raw['coordinates.longitude'], df_raw['coordinates.latitude']),
        crs="EPSG:4326"
    )
    
    # Recompute real-time operational status logic
    gdf_cleaned['datetimeLast.utc'] = pd.to_datetime(gdf_cleaned['datetimeLast.utc'], errors='coerce')
    network_baseline_date = gdf_cleaned['datetimeLast.utc'].max()
    gdf_cleaned['days_since_last_seen'] = (network_baseline_date - gdf_cleaned['datetimeLast.utc']).dt.days

    def classify_sensor_status(days):
        if pd.isna(days): return 'Unknown'
        if days <= 5: return 'Operational'
        elif days <= 30: return 'Unreliable'
        else: return 'Failed'

    gdf_cleaned['status'] = gdf_cleaned['days_since_last_seen'].apply(classify_sensor_status)
    print("✅ Base sensor network memory successfully restored!")

    # ==========================================
    # STEP 2: EXTRACT WORLDPOP STUDENT DEMOGRAPHICS
    # ==========================================
    gdf_cleaned['school_age_pop_density'] = 0.0
    age_folder = "age"
    demographic_files = [
        "lao_f_05_2026_CN_100m_R2025A_v1.tif", "lao_f_10_2026_CN_100m_R2025A_v1.tif", "lao_f_15_2026_CN_100m_R2025A_v1.tif",
        "lao_m_05_2026_CN_100m_R2025A_v1.tif", "lao_m_10_2026_CN_100m_R2025A_v1.tif", "lao_m_15_2026_CN_100m_R2025A_v1.tif"
    ]
    
    print("\n🌲 Sampling WorldPop demographic layers...")
    for file_name in demographic_files:
        file_path = os.path.join(age_folder, file_name)
        if not os.path.exists(file_path):
            print(f"  ⚠️ Skipping missing file: {file_name}")
            continue
        with rasterio.open(file_path) as src:
            gdf_temp = gdf_cleaned.to_crs(src.crs)
            coords = zip(gdf_temp.geometry.x, gdf_temp.geometry.y)
            pixel_values = [max(0.0, float(val[0])) for val in src.sample(coords)]
            gdf_cleaned['school_age_pop_density'] += pixel_values
            
    median_pop = gdf_cleaned['school_age_pop_density'].median()
    gdf_cleaned['student_density_group'] = gdf_cleaned['school_age_pop_density'].apply(
        lambda x: 'High Student Density' if x > median_pop else 'Low Student Density'
    )
    print("🎯 WorldPop demographic cohort sums complete.")

    # ==========================================
    # STEP 3: PROXIMITY TO OPENSTREETMAP SCHOOLS
    # ==========================================
    fiona.drvsupport.supported_drivers['KML'] = 'rw'
    kml_path = "schools.kml"
    
    if not os.path.exists(kml_path):
        print(f"\n❌ Vector Error: '{kml_path}' not found!")
    else:
        print("\n🏫 Processing OpenStreetMap school locations...")
        schools_gdf = gpd.read_file(kml_path, driver='KML')
        
        # Reproject to metric system (Mollweide) for real distance metrics
        gdf_metric = gdf_cleaned.to_crs("ESRI:54009")
        schools_metric = schools_gdf.to_crs("ESRI:54009")
        
        distances = []
        for geom in gdf_metric.geometry:
            min_dist_meters = schools_metric.distance(geom).min()
            distances.append(min_dist_meters / 1000.0) # Meters to km
            
        gdf_cleaned['distance_to_nearest_school_km'] = distances
        gdf_cleaned['school_proximity_zone'] = gdf_cleaned['distance_to_nearest_school_km'].apply(
            lambda x: 'Inside School Zone (<1.5km)' if x <= 1.5 else 'Outside School Zone'
        )
        print("🎯 Distance calculations complete!")

        # ==========================================
        # STEP 4: GENERATE EVALUATION CROSS-TABS
        # ==========================================
        print("\n📊 CROSS-TABULATION A: SENSOR RELIABILITY BY STUDENT POPULATION DENSITY (WORLDPOP)")
        print("==================================================================================")
        ct_pop = pd.crosstab(gdf_cleaned['student_density_group'], gdf_cleaned['status'], normalize='index') * 100
        print(ct_pop.round(1).to_string())
        print("==================================================================================")

        print("\n📊 CROSS-TABULATION B: SENSOR RELIABILITY BY PHYSICAL SCHOOL PROXIMITY (OSM KML)")
        print("==================================================================================")
        ct_school = pd.crosstab(gdf_cleaned['school_proximity_zone'], gdf_cleaned['status'], normalize='index') * 100
        print(ct_school.round(1).to_string())
        print("==================================================================================")
        
        print("\n📍 Distribution of active tracking nodes near schools:")
        print(gdf_cleaned['school_proximity_zone'].value_counts())

🎒 Initializing Consolidated Vulnerability Pipeline...
✅ Base sensor network memory successfully restored!

🌲 Sampling WorldPop demographic layers...
🎯 WorldPop demographic cohort sums complete.

🏫 Processing OpenStreetMap school locations...
🎯 Distance calculations complete!

📊 CROSS-TABULATION A: SENSOR RELIABILITY BY STUDENT POPULATION DENSITY (WORLDPOP)
status                 Failed  Operational  Unreliable
student_density_group                                 
High Student Density     13.6         75.3        11.1
Low Student Density      20.7         70.7         8.5

📊 CROSS-TABULATION B: SENSOR RELIABILITY BY PHYSICAL SCHOOL PROXIMITY (OSM KML)
status                       Failed  Operational  Unreliable
school_proximity_zone                                       
Inside School Zone (<1.5km)    20.8         68.8        10.4
Outside School Zone            15.7         74.8         9.6

📍 Distribution of active tracking nodes near schools:
school_proximity_zone
Outside School Zone